In [1]:
import pandas as pd
import numpy as np
import plotly.graph_objects as go
import plotly.express as px
import warnings
import os
warnings.filterwarnings("ignore")

os.chdir(r"E:\retailpulse")

df = pd.read_csv("data/processed/master.csv",
                 parse_dates=["order_date"])

# Aggregate to weekly revenue
weekly = (df.groupby(df["order_date"].dt.to_period("W"))
            .agg(revenue=("revenue","sum"),
                 orders=("order_id","nunique"))
            .reset_index())
weekly["order_date"] = weekly["order_date"].dt.start_time
weekly = weekly.sort_values("order_date").reset_index(drop=True)

# Remove first and last incomplete weeks
weekly = weekly.iloc[1:-1]

print(f"Weekly shape: {weekly.shape}")
print(f"Date range: {weekly['order_date'].min()} "
      f"→ {weekly['order_date'].max()}")
print(f"\nSample:\n{weekly.head()}")

Weekly shape: (89, 3)
Date range: 2016-10-03 00:00:00 → 2018-08-20 00:00:00

Sample:
  order_date   revenue  orders
1 2016-10-03  42593.93     231
2 2016-10-10   3896.73      34
3 2016-12-19     19.62       1
4 2017-01-02   3651.39      44
5 2017-01-09  12582.30      73


In [2]:
# Sort and create lag features
weekly = weekly.sort_values("order_date").reset_index(drop=True)

# Lag features
for lag in [1, 2, 4, 8]:
    weekly[f"lag_{lag}"] = weekly["revenue"].shift(lag)

# Rolling features
for window in [4, 8]:
    weekly[f"roll_mean_{window}"] = (weekly["revenue"]
                                     .shift(1)
                                     .rolling(window, min_periods=2)
                                     .mean())
    weekly[f"roll_std_{window}"]  = (weekly["revenue"]
                                     .shift(1)
                                     .rolling(window, min_periods=2)
                                     .std())

# Calendar features
weekly["week_of_year"] = weekly["order_date"].dt.isocalendar().week.astype(int)
weekly["month"]        = weekly["order_date"].dt.month
weekly["quarter"]      = weekly["order_date"].dt.quarter

# Drop NaN rows from lag creation
weekly_clean = weekly.dropna().reset_index(drop=True)

print(f"Clean weekly shape: {weekly_clean.shape}")
print(f"Features: {list(weekly_clean.columns)}")

Clean weekly shape: (81, 14)
Features: ['order_date', 'revenue', 'orders', 'lag_1', 'lag_2', 'lag_4', 'lag_8', 'roll_mean_4', 'roll_std_4', 'roll_mean_8', 'roll_std_8', 'week_of_year', 'month', 'quarter']


In [3]:
import lightgbm as lgb
from sklearn.metrics import mean_absolute_error

FEATURES = ([f"lag_{l}" for l in [1,2,4,8]] +
            [f"roll_mean_{w}" for w in [4,8]] +
            [f"roll_std_{w}"  for w in [4,8]] +
            ["week_of_year","month","quarter"])

TARGET = "revenue"

# Time-based split — last 8 weeks as test
cutoff = weekly_clean["order_date"].max() - pd.Timedelta(weeks=8)
train  = weekly_clean[weekly_clean["order_date"] <= cutoff]
test   = weekly_clean[weekly_clean["order_date"] >  cutoff]

X_train = train[FEATURES]
y_train = train[TARGET]
X_test  = test[FEATURES]
y_test  = test[TARGET]

print(f"Train: {len(X_train)} weeks")
print(f"Test:  {len(X_test)} weeks")

# Naive baseline — last week's value
naive_preds = y_test.shift(1).fillna(y_test.iloc[0])

# LightGBM model
model = lgb.LGBMRegressor(
    n_estimators=200,
    learning_rate=0.05,
    num_leaves=16,
    random_state=42,
    verbose=-1)

model.fit(X_train, y_train)
preds = model.predict(X_test).clip(min=0)

# Evaluate
def wape(actual, predicted):
    return np.abs(actual - predicted).sum() / actual.sum()

model_wape = wape(y_test.values, preds)
naive_wape = wape(y_test.values, naive_preds.values)

print("\n" + "="*45)
print("  FORECAST RESULTS")
print("="*45)
print(f"  Naive WAPE:  {naive_wape:.4f} ({naive_wape*100:.1f}%)")
print(f"  Model WAPE:  {model_wape:.4f} ({model_wape*100:.1f}%)")
print(f"  Improvement: {(naive_wape-model_wape)/naive_wape*100:.1f}%")
if model_wape < naive_wape:
    print("  🎉 Model beats baseline!")
else:
    print("  📊 Using naive baseline")

Train: 73 weeks
Test:  8 weeks

  FORECAST RESULTS
  Naive WAPE:  0.1614 (16.1%)
  Model WAPE:  0.2512 (25.1%)
  Improvement: -55.7%
  📊 Using naive baseline


In [4]:
import joblib

# Plot
fig = go.Figure()
fig.add_trace(go.Scatter(
    x=weekly_clean["order_date"],
    y=weekly_clean["revenue"],
    name="Actual", line=dict(color="#2563eb", width=2)))

fig.add_trace(go.Scatter(
    x=test["order_date"],
    y=preds,
    name="LightGBM Forecast",
    line=dict(color="#f59e0b", width=2, dash="dash")))

fig.add_trace(go.Scatter(
    x=test["order_date"],
    y=naive_preds.values,
    name="Naive Baseline",
    line=dict(color="#ef4444", width=2, dash="dot")))

fig.add_vline(x=cutoff, line_dash="dot",
              line_color="gray",
              annotation_text="Forecast Start")

fig.update_layout(
    title="Weekly Revenue Forecast vs Actual",
    xaxis_title="Week",
    yaxis_title="Revenue (BRL)")
fig.show()

# Save
weekly_clean.to_csv("data/processed/weekly_features.csv", index=False)
joblib.dump(model,    "data/processed/forecast_model.pkl")
joblib.dump(FEATURES, "data/processed/forecast_features.pkl")

# Save forecast results
forecast_results = test[["order_date","revenue"]].copy()
forecast_results["prediction"] = preds
forecast_results.to_csv("data/processed/forecast_results.csv",
                         index=False)

print("✅ Forecast model saved!")

✅ Forecast model saved!


In [5]:
# Naive baseline wins — use it as our official forecast
# Predict next 8 weeks using last known values

last_week = weekly_clean["order_date"].max()
last_revenue = weekly_clean["revenue"].iloc[-1]
avg_recent = weekly_clean["revenue"].tail(8).mean()

future_weeks = pd.date_range(
    start=last_week + pd.Timedelta(weeks=1),
    periods=8, freq="W")

# Naive forecast = rolling average of last 4 weeks
last_4_avg = weekly_clean["revenue"].tail(4).mean()

future_forecast = pd.DataFrame({
    "order_date": future_weeks,
    "predicted_revenue": [last_4_avg] * 8,
    "type": "Forecast"
})

print("📅 Next 8 Weeks Forecast:")
print(future_forecast.to_string(index=False))

# Plot historical + forecast
fig = go.Figure()

fig.add_trace(go.Scatter(
    x=weekly_clean["order_date"],
    y=weekly_clean["revenue"],
    name="Historical Revenue",
    line=dict(color="#2563eb", width=2)))

fig.add_trace(go.Scatter(
    x=future_forecast["order_date"],
    y=future_forecast["predicted_revenue"],
    name="Forecast (Next 8 Weeks)",
    line=dict(color="#f59e0b", width=2, dash="dash")))

fig.add_vline(x=last_week,
              line_dash="dot", line_color="gray",
              annotation_text="Today")

fig.update_layout(
    title="Revenue Forecast — Next 8 Weeks",
    xaxis_title="Week",
    yaxis_title="Revenue (BRL)")
fig.show()

# Save future forecast
future_forecast.to_csv(
    "data/processed/future_forecast.csv", index=False)
weekly_clean.to_csv(
    "data/processed/weekly_features.csv", index=False)

print(f"\n✅ Forecast saved!")
print(f"   Avg weekly revenue (last 4 weeks): "
      f"BRL {last_4_avg:,.0f}")
print(f"   Forecast for next 8 weeks:         "
      f"BRL {last_4_avg*8:,.0f}")

📅 Next 8 Weeks Forecast:
order_date  predicted_revenue     type
2018-09-02          267607.57 Forecast
2018-09-09          267607.57 Forecast
2018-09-16          267607.57 Forecast
2018-09-23          267607.57 Forecast
2018-09-30          267607.57 Forecast
2018-10-07          267607.57 Forecast
2018-10-14          267607.57 Forecast
2018-10-21          267607.57 Forecast



✅ Forecast saved!
   Avg weekly revenue (last 4 weeks): BRL 267,608
   Forecast for next 8 weeks:         BRL 2,140,861
